# ⬡ SkillChain — Explorer la blockchain

Ce notebook est votre fil rouge pour le workshop.
Chaque cellule explore un concept différent de la blockchain **en lisant des données réelles** sur le réseau Sepolia.

**Comment l'utiliser :**
- Exécutez les cellules **dans l'ordre** (▶ ou Shift+Entrée)
- Lisez les commentaires avant chaque cellule
- Quand vous voyez `# ✏️ MODIFIEZ ICI`, changez la valeur indiquée

---
> **Réseau utilisé :** Sepolia Testnet (clone d'Ethereum, ETH sans valeur réelle)  
> **Contrat SkillBadge :** déployé et actif sur ce réseau

## 0. Installation des dépendances

On installe `web3.py` — la librairie Python pour interagir avec Ethereum.  
C'est l'équivalent Python de `ethers.js` utilisé dans la DApp.

In [ ]:
!pip install web3 -q
print("✓ web3.py installé")

## 1. Connexion au réseau Sepolia

On se connecte à Sepolia via un **nœud RPC public** — un serveur qui relaie nos requêtes vers le réseau.

On n'a pas besoin de wallet ni de clé privée pour **lire** la blockchain.
La lecture est publique et gratuite.

In [ ]:
from web3 import Web3
from datetime import datetime, timezone

# Nœud RPC public Sepolia — aucune clé API requise
RPC_URL = "https://ethereum-sepolia.publicnode.com"
w3 = Web3(Web3.HTTPProvider(RPC_URL))

if w3.is_connected():
    bloc = w3.eth.block_number
    print(f"✓ Connecté à Sepolia")
    print(f"  Numéro du dernier bloc : {bloc:,}")
    print(f"  Chaque bloc = ~12 secondes de transactions validées")
else:
    print("✗ Connexion échouée — vérifiez votre connexion internet")

## 2. Charger le contrat SkillBadge

Pour interagir avec un smart contract, on a besoin de deux choses :
- Son **adresse** sur le réseau (où il est déployé)
- Son **ABI** (Application Binary Interface) : le schéma de ses fonctions

L'ABI dit à web3.py comment encoder nos appels de fonctions en bytecode EVM.

---
> ✏️ **Si vous avez déployé votre propre contrat au Bloc 3**, remplacez l'adresse ci-dessous.

In [ ]:
# ✏️ MODIFIEZ ICI si vous avez votre propre contrat déployé
CONTRACT_ADDRESS = "0x679E528c0712cA4F1Dbe37Ab99Dfd66855091fa8"

# ABI — description des fonctions du contrat
ABI = [
    {
        "inputs": [],
        "name": "getBadgeCount",
        "outputs": [{"internalType": "uint256", "name": "", "type": "uint256"}],
        "stateMutability": "view",
        "type": "function"
    },
    {
        "inputs": [{"internalType": "address", "name": "student", "type": "address"}],
        "name": "getBadgesOf",
        "outputs": [{"internalType": "uint256[]", "name": "", "type": "uint256[]"}],
        "stateMutability": "view",
        "type": "function"
    },
    {
        "inputs": [{"internalType": "uint256", "name": "index", "type": "uint256"}],
        "name": "getBadge",
        "outputs": [
            {"internalType": "address", "name": "", "type": "address"},
            {"internalType": "string",  "name": "", "type": "string"},
            {"internalType": "uint256", "name": "", "type": "uint256"},
            {"internalType": "address", "name": "", "type": "address"}
        ],
        "stateMutability": "view",
        "type": "function"
    }
]

# Web3 exige une adresse en checksum (majuscules/minuscules précis)
address = Web3.to_checksum_address(CONTRACT_ADDRESS)
contract = w3.eth.contract(address=address, abi=ABI)

print(f"✓ Contrat chargé")
print(f"  Adresse : {address}")
print(f"  Etherscan : https://sepolia.etherscan.io/address/{address}")

## 3. Combien de badges existent sur la blockchain ?

On appelle `getBadgeCount()` — une fonction `view` du contrat.  
Pas de transaction, pas de gas, pas de wallet. Juste une lecture.

In [ ]:
count = contract.functions.getBadgeCount().call()

print(f"Nombre de badges certifiés sur la blockchain : {count}")

if count == 0:
    print("  (Aucun badge pour l'instant — émettez-en un via la DApp d'abord)")

## 4. Lire tous les badges

On parcourt chaque badge et on affiche ses données.  
Ces données sont **stockées sur la blockchain** — elles ne peuvent pas être modifiées ou supprimées.  

Notez le `block.timestamp` : c'est l'horodatage fourni par le validateur du bloc au moment de la transaction.

In [ ]:
def fmt_addr(addr):
    return f"{addr[:6]}...{addr[-4:]}"

def fmt_time(ts):
    return datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%d/%m/%Y %H:%M UTC")

count = contract.functions.getBadgeCount().call()

if count == 0:
    print("Aucun badge trouvé. Émettez-en un via la DApp puis relancez cette cellule.")
else:
    print(f"{'#':<4} {'Compétence':<28} {'Étudiant':<14} {'Date':<22} {'Émetteur':<14}")
    print("─" * 90)
    for i in range(count):
        student, skill, issued_at, issuer = contract.functions.getBadge(i).call()
        print(f"{i:<4} {skill:<28} {fmt_addr(student):<14} {fmt_time(issued_at):<22} {fmt_addr(issuer):<14}")

## 5. Vérifier les badges d'une adresse spécifique

Entrez votre adresse MetaMask pour voir vos badges.

> ✏️ **Remplacez** `0x...` par votre adresse MetaMask (celle visible dans l'extension)

In [ ]:
# ✏️ MODIFIEZ ICI — votre adresse MetaMask
MON_ADRESSE = "0x"  # ex: "0x287896730Ff677Da0dD9D1ccBD103E84D9A7e4eF"

if MON_ADRESSE == "0x" or len(MON_ADRESSE) != 42:
    print("⚠️  Entrez votre adresse MetaMask complète (42 caractères, commence par 0x)")
else:
    checksum = Web3.to_checksum_address(MON_ADRESSE)
    indexes = contract.functions.getBadgesOf(checksum).call()

    print(f"Adresse  : {checksum}")
    print(f"Badges   : {len(indexes)}")
    print()

    if len(indexes) == 0:
        print("  Aucun badge pour cette adresse.")
        print("  → Émettez un badge via la DApp en utilisant cette adresse comme destinataire.")
    else:
        for idx in indexes:
            student, skill, issued_at, issuer = contract.functions.getBadge(idx).call()
            print(f"  Badge #{idx}")
            print(f"    Compétence : {skill}")
            print(f"    Émis le    : {fmt_time(issued_at)}")
            print(f"    Par        : {issuer}")
            print()

## 6. Explorer un bloc

Un bloc = un ensemble de transactions validées à un instant donné.  
Chaque bloc référence le hash du bloc précédent — c'est ce qui forme la **chaîne**.

In [ ]:
# On lit le dernier bloc
bloc = w3.eth.get_block('latest')

print(f"Bloc #{bloc['number']:,}")
print(f"  Timestamp     : {fmt_time(bloc['timestamp'])}")
print(f"  Hash          : {bloc['hash'].hex()}")
print(f"  Hash précédent: {bloc['parentHash'].hex()}")
print(f"  Transactions  : {len(bloc['transactions'])}")
print(f"  Gas utilisé   : {bloc['gasUsed']:,}")
print(f"  Gas limite    : {bloc['gasLimit']:,}")
print()
print("→ 'parentHash' est le hash du bloc précédent.")
print("  Modifier un ancien bloc changerait son hash, cassant ce lien — et tous les suivants.")

## 7. Explorer une transaction

Chaque émission de badge crée une transaction sur la blockchain.  
On peut lire exactement ce qui s'est passé : qui a appelé quoi, quand, combien ça a coûté.

> ✏️ **Remplacez** `0x...` par le hash d'une transaction visible sur Etherscan  
> (récupérez-le depuis la DApp après avoir émis un badge — le lien "Voir sur Etherscan" donne le hash)

In [ ]:
# ✏️ MODIFIEZ ICI — hash d'une transaction issueBadge
TX_HASH = "0x"  # ex: "0xabc123..."

if TX_HASH == "0x" or len(TX_HASH) < 60:
    print("⚠️  Entrez un hash de transaction valide (66 caractères, commence par 0x)")
    print("    Récupérez-le depuis la DApp après avoir émis un badge.")
else:
    tx      = w3.eth.get_transaction(TX_HASH)
    receipt = w3.eth.get_transaction_receipt(TX_HASH)
    bloc    = w3.eth.get_block(tx['blockNumber'])

    statut = "✓ Succès" if receipt['status'] == 1 else "✗ Échec"

    print(f"Transaction")
    print(f"  Hash         : {TX_HASH}")
    print(f"  Statut       : {statut}")
    print(f"  Bloc         : #{tx['blockNumber']:,}")
    print(f"  Timestamp    : {fmt_time(bloc['timestamp'])}")
    print()
    print(f"Expéditeur")
    print(f"  De           : {tx['from']}  ← msg.sender dans le contrat")
    print(f"  Vers contrat : {tx['to']}")
    print()
    print(f"Coût")
    print(f"  Gas utilisé  : {receipt['gasUsed']:,} unités")
    gas_price_gwei = tx['gasPrice'] / 1e9
    cout_eth = (receipt['gasUsed'] * tx['gasPrice']) / 1e18
    print(f"  Gas price    : {gas_price_gwei:.4f} Gwei")
    print(f"  Coût total   : {cout_eth:.8f} ETH")
    print()
    print(f"  → Sur mainnet à 20 Gwei ≈ {receipt['gasUsed'] * 20 / 1e9:.6f} ETH ≈ 3-5€")

## 8. Le hash — démonstration

Le hash est la brique fondamentale de la blockchain.  
Ethereum utilise **keccak256** (variante de SHA-3).

Propriétés démontrées ci-dessous :
1. **Déterministe** — même entrée → même sortie, toujours
2. **Effet avalanche** — 1 caractère change → hash totalement différent
3. **Taille fixe** — peu importe la taille de l'entrée, le hash fait toujours 32 octets

In [ ]:
from eth_hash.auto import keccak

def show_hash(label, text):
    h = keccak(text.encode('utf-8')).hex()
    print(f"  {label:<30} →  0x{h}")

print("Propriété 1 — Déterministe")
show_hash('"Bonjour" (1ère fois)', "Bonjour")
show_hash('"Bonjour" (2ème fois)', "Bonjour")
print("  → Identiques. Toujours.")
print()

print("Propriété 2 — Effet avalanche")
show_hash('"Bonjour"  (B majuscule)', "Bonjour")
show_hash('"bonjour"  (b minuscule)', "bonjour")
print("  → 1 seul caractère change → hash méconnaissable")
print()

print("Propriété 3 — Taille fixe")
show_hash('"A"  (1 caractère)',    "A")
show_hash('"Solidity Basics"',     "Solidity Basics")
show_hash('"A" * 10000 (10k car.)', "A" * 10000)
print("  → Toujours 64 caractères hex = 32 octets = 256 bits")

## 9. Comment votre adresse MetaMask est dérivée

Une adresse Ethereum n'est pas aléatoire — elle est **dérivée mathématiquement** de votre clé privée.  
La cellule ci-dessous montre comment, **à partir d'une clé privée de test** (jamais utiliser une vraie clé ici).

In [ ]:
from eth_account import Account

# ⚠️ CLÉ DE DÉMONSTRATION UNIQUEMENT — ne jamais entrer une vraie clé privée dans un notebook
CLE_DEMO = "0xaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa"

compte = Account.from_key(CLE_DEMO)

print("Dérivation d'une adresse Ethereum")
print()
print(f"  Clé privée  : {CLE_DEMO}")
print(f"  (256 bits aléatoires — c'est votre secret absolu)")
print()
print(f"  Adresse     : {compte.address}")
print(f"  (keccak256 de la clé publique ECDSA, 20 derniers octets)")
print()
print("Vérification — la même clé donne toujours la même adresse :")
compte2 = Account.from_key(CLE_DEMO)
print(f"  Identique ? {compte.address == compte2.address}")
print()
print("→ Dans MetaMask, votre clé privée est stockée chiffrée localement.")
print("  La seed phrase (12 mots) permet de la régénérer — ne la partagez jamais.")

## 10. Vérifier la santé du réseau

Quelques métriques globales du réseau Sepolia — pour observer que c'est un réseau vivant, pas un serveur simulé.

In [ ]:
dernier_bloc   = w3.eth.get_block('latest')
bloc_precedent = w3.eth.get_block(dernier_bloc['number'] - 1)

temps_bloc = dernier_bloc['timestamp'] - bloc_precedent['timestamp']
nb_tx      = len(dernier_bloc['transactions'])

print("État du réseau Sepolia")
print(f"  Bloc actuel            : #{dernier_bloc['number']:,}")
print(f"  Temps entre les blocs  : {temps_bloc} secondes  (cible : ~12s sur PoS)")
print(f"  Tx dans ce bloc        : {nb_tx}")
print(f"  Gas utilisé            : {dernier_bloc['gasUsed']:,}")
print(f"  Base fee               : {dernier_bloc['baseFeePerGas'] / 1e9:.4f} Gwei")
print()
print(f"  Votre contrat SkillBadge est stocké dans l'un de ces {dernier_bloc['number']:,} blocs.")
print(f"  Il y restera pour toujours.")

---

## Récapitulatif — ce que vous avez fait dans ce notebook

| Cellule | Concept exploré |
|---|---|
| 1 | Connexion à un réseau décentralisé via RPC |
| 2 | ABI : schéma d'un smart contract |
| 3-5 | Lecture de données on-chain (view functions, gratuites) |
| 6 | Structure d'un bloc et chaînage via parentHash |
| 7 | Anatomie d'une transaction : gas, coût, sender |
| 8 | Hash keccak256 : déterminisme, effet avalanche, taille fixe |
| 9 | Dérivation d'une adresse depuis une clé privée |
| 10 | Réseau vivant : temps de bloc, base fee, PoS |

---

> **Prochaine étape :** ouvrir la DApp SkillChain pour **émettre** des badges (nécessite MetaMask — les transactions écrivent sur la blockchain et coûtent du gas).